# Hospital 0 — Urban Teaching Hospital
**FL-Pneumonia-Detection | Distributed Simulation**

Non-IID pneumonia prevalence target: **70%**

### Per-round workflow
1. Download current global model from WandB
2. Train locally (with Opacus DP)
3. Upload local weights back to WandB
4. Run `fl_aggregator.ipynb` with the same `ROUND_NUM`

> **Only change `ROUND_NUM`** before each round. Everything else is fixed.

In [ ]:
!pip install opacus==1.4.0 wandb --quiet
print('Packages ready')

In [ ]:
import os, json, random
from collections import OrderedDict
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import wandb
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from kaggle_secrets import UserSecretsClient

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HOSPITAL_ID   = 0   # fixed
HOSPITAL_NAME = 'Urban Teaching Hospital'
PNEUMONIA_RATIO = 0.7  # non-IID target for this hospital

# ── UPDATE THIS BEFORE EACH ROUND ──
ROUND_NUM = 1
# ───────────────────────────────────

FL_CONFIG = {
    'num_rounds':     10,
    'num_clients':    3,
    'local_epochs':   3,
    'learning_rate':  1e-3,
    'lr_decay':       0.95,
    'batch_size':     32,
    'use_dp':         True,
    'target_epsilon': 10.0,
    'target_delta':   1e-5,
    'max_grad_norm':  1.0,
}

DATA_ROOT         = '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray'
WORK_DIR          = '/kaggle/working'
WANDB_PROJECT     = 'fl-pneumonia-detection'
GLOBAL_ARTIFACT   = 'global-model'
HOSPITAL_ARTIFACT = 'hospital-weights'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Hospital 0 | Round {ROUND_NUM} | Device: {device}')


In [ ]:
wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
wandb.login(key=wandb_key)
print('WandB logged in')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class ChestXrayDataset(Dataset):
    CLASSES = ['NORMAL', 'PNEUMONIA']
    def __init__(self, data_dir=None, transform=None, images=None, labels=None):
        self.transform = transform
        if images is not None:
            self.images, self.labels = list(images), list(labels)
        else:
            self.images, self.labels = [], []
            for label, cls in enumerate(self.CLASSES):
                cls_dir = os.path.join(data_dir, cls)
                if not os.path.isdir(cls_dir): continue
                for fname in sorted(os.listdir(cls_dir)):
                    if fname.lower().endswith(('.jpg','.jpeg','.png')):
                        self.images.append(os.path.join(cls_dir, fname))
                        self.labels.append(label)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return (self.transform(img) if self.transform else img), self.labels[idx]

def get_train_transforms():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

full_train = ChestXrayDataset(os.path.join(DATA_ROOT, 'train'))
print(f'Full train set loaded: {len(full_train)} images')


In [ ]:
def get_hospital_partition(dataset, hospital_id, num_hospitals=3, seed=42):
    """
    Deterministic non-IID partition.
    All hospital notebooks use the same function + seed so each hospital
    always gets the same indices, regardless of run order.
    """
    ratios = [0.70, 0.80, 0.60]
    rng = np.random.default_rng(seed)
    normal_idx    = [i for i, l in enumerate(dataset.labels) if l == 0]
    pneumonia_idx = [i for i, l in enumerate(dataset.labels) if l == 1]
    rng.shuffle(normal_idx)
    rng.shuffle(pneumonia_idx)
    hospital_size = len(dataset) // num_hospitals
    n_ptr, p_ptr = 0, 0
    for hid in range(num_hospitals):
        p_count = int(hospital_size * ratios[hid])
        n_count = hospital_size - p_count
        if hid == num_hospitals - 1:
            n_count = len(normal_idx)    - n_ptr
            p_count = len(pneumonia_idx) - p_ptr
        if hid == hospital_id:
            h_idx = normal_idx[n_ptr:n_ptr + n_count] + pneumonia_idx[p_ptr:p_ptr + p_count]
            rng2 = np.random.default_rng(seed + hid)
            rng2.shuffle(h_idx)
            return h_idx
        n_ptr += n_count
        p_ptr += p_count

my_indices = get_hospital_partition(full_train, HOSPITAL_ID)
my_labels  = [full_train.labels[i] for i in my_indices]
print(f'Hospital {HOSPITAL_ID} partition: {len(my_indices)} samples')
print(f'  NORMAL={my_labels.count(0)}  PNEUMONIA={my_labels.count(1)}')
print(f'  Pneumonia rate: {my_labels.count(1)/len(my_labels):.1%} (target {PNEUMONIA_RATIO:.0%})')

my_dataset = ChestXrayDataset(
    images=[full_train.images[i] for i in my_indices],
    labels=my_labels,
    transform=get_train_transforms(),
)
train_loader = DataLoader(
    my_dataset, batch_size=FL_CONFIG['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True,
)
print(f'  DataLoader: {len(train_loader)} batches/epoch')


In [ ]:
def create_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, 2),
    )
    return ModuleValidator.fix(model)

print(f'Downloading {GLOBAL_ARTIFACT}:round-{ROUND_NUM - 1} from WandB...')
api      = wandb.Api()
artifact = api.artifact(f'{WANDB_PROJECT}/{GLOBAL_ARTIFACT}:round-{ROUND_NUM - 1}', type='model')
art_dir  = artifact.download(root=f'{WORK_DIR}/global_model')
ckpt     = torch.load(os.path.join(art_dir, 'global_model.pth'), map_location='cpu')
model    = create_model()
model.load_state_dict(ckpt['model_state_dict'])
model    = model.to(device)
print(f'Global model (round {ROUND_NUM - 1}) loaded')


In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    name=f'hospital-{HOSPITAL_ID}-round-{ROUND_NUM}',
    job_type='local-training',
    config={**FL_CONFIG, 'hospital_id': HOSPITAL_ID,
            'hospital_name': HOSPITAL_NAME, 'round': ROUND_NUM,
            'n_samples': len(my_dataset)},
)

lr        = FL_CONFIG['learning_rate'] * (FL_CONFIG['lr_decay'] ** ROUND_NUM)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
epsilon_used = None

if FL_CONFIG['use_dp']:
    eps_per_round  = FL_CONFIG['target_epsilon'] / FL_CONFIG['num_rounds']
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
        module=model, optimizer=optimizer, data_loader=train_loader,
        target_epsilon=eps_per_round, target_delta=FL_CONFIG['target_delta'],
        epochs=FL_CONFIG['local_epochs'], max_grad_norm=FL_CONFIG['max_grad_norm'],
    )
    print(f'Opacus PrivacyEngine attached (eps_per_round={eps_per_round:.2f})')

print(f'Training {FL_CONFIG["local_epochs"]} local epochs...')
model.train()
total_loss, correct, total = 0.0, 0, 0

for epoch in range(FL_CONFIG['local_epochs']):
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        ep_loss    += loss.item() * images.size(0)
        ep_correct += (out.argmax(1) == labels).sum().item()
        ep_total   += labels.size(0)
    print(f'  Epoch {epoch+1}: loss={ep_loss/ep_total:.4f} acc={ep_correct/ep_total:.4f}')
    total_loss += ep_loss; correct += ep_correct; total += ep_total

avg_loss = total_loss / total
avg_acc  = correct    / total

if FL_CONFIG['use_dp']:
    epsilon_used = privacy_engine.get_epsilon(delta=FL_CONFIG['target_delta'])
    print(f'Epsilon consumed this round: {epsilon_used:.4f}')

wandb.log({'round': ROUND_NUM, 'train/loss': avg_loss,
           'train/accuracy': avg_acc, 'privacy/epsilon': epsilon_used})
print(f'Loss={avg_loss:.4f} | Acc={avg_acc:.4f}')


In [ ]:
ckpt_path = f'{WORK_DIR}/hospital_{HOSPITAL_ID}_round_{ROUND_NUM}.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'hospital_id':      HOSPITAL_ID,
    'hospital_name':    HOSPITAL_NAME,
    'round':            ROUND_NUM,
    'n_samples':        len(my_dataset),
    'train_loss':       avg_loss,
    'train_accuracy':   avg_acc,
    'epsilon':          epsilon_used,
}, ckpt_path)

artifact = wandb.Artifact(
    name=HOSPITAL_ARTIFACT, type='model',
    metadata={'hospital_id': HOSPITAL_ID, 'round': ROUND_NUM,
              'n_samples': len(my_dataset), 'train_accuracy': float(avg_acc),
              'epsilon': float(epsilon_used) if epsilon_used else None},
)
artifact.add_file(ckpt_path, name=f'hospital_{HOSPITAL_ID}.pth')
run.log_artifact(artifact, aliases=[f'h{HOSPITAL_ID}-round-{ROUND_NUM}'])
wandb.finish()
print(f'Weights uploaded: "{HOSPITAL_ARTIFACT}:h{HOSPITAL_ID}-round-{ROUND_NUM}"')
print(f'Next: run fl_aggregator.ipynb with ROUND_NUM={ROUND_NUM}')
